# normalize_ghsa
`stg_ghsa_valid` → 날짜 / Severity / Identifier를 표준 스키마로 정규화한 `stg_ghsa_normalized`.
서로 다른 GHSA API 버전(snake_case ↔ camelCase, 여러 CVSS 경로 등)의 필드 후보 중 **먼저 존재하는 값**을 선택한다(02_Bronze_Silver_저장구조.md 표 1 그대로).

In [ ]:
%run 00_silver_common

In [ ]:
from pyspark.sql.types import ArrayType, StructType

raw_df = spark.table(TBL["stg_ghsa_valid"])
# nested_or_none()/nested_coalesce()는 00_silver_common에서 상속 (KEV 이벤트 payload 파싱과 공용)


In [ ]:
ghsa_id = F.upper(F.trim(coalesce_existing(raw_df, ["ghsa_id", "id"])))
summary = coalesce_existing(raw_df, ["summary", "title"])
description = coalesce_existing(raw_df, ["description", "details"])
severity = existing_col(raw_df, "severity")
published_at = coalesce_existing(raw_df, ["published_at", "publishedAt"])
updated_at = coalesce_existing(raw_df, ["updated_at", "updatedAt"])
withdrawn_at = coalesce_existing(raw_df, ["withdrawn_at", "withdrawnAt"])
html_url = coalesce_existing(raw_df, ["html_url", "url"])

# cwes: 원소가 문자열이거나 {cwe_id/id/cweId: ...} 객체일 수 있음 -> CWE ID 문자열 배열만 추출
if "cwes" in raw_df.columns and isinstance(raw_df.schema["cwes"].dataType, ArrayType) \
        and isinstance(raw_df.schema["cwes"].dataType.elementType, StructType):
    elem_fields = raw_df.schema["cwes"].dataType.elementType.fieldNames()
    cwe_key = next((k for k in ("cwe_id", "id", "cweId") if k in elem_fields), None)
    cwe_ids = F.transform(F.col("cwes"), lambda x: x[cwe_key]) if cwe_key else F.array().cast("array<string>")
else:
    cwe_ids = existing_array_col(raw_df, "cwes")

cvss_score = F.coalesce(
    nested_or_none(raw_df, "cvss.score"),
    nested_or_none(raw_df, "cvss_v3.score"),
    existing_col(raw_df, "cvss_score"),
    existing_col(raw_df, "cvssScore"),
).cast(DoubleType())

cvss_vector = F.coalesce(
    nested_or_none(raw_df, "cvss.vector_string"),
    nested_or_none(raw_df, "cvss.vectorString"),
    nested_or_none(raw_df, "cvss_v3.vector_string"),
    nested_or_none(raw_df, "cvss_v3.vectorString"),
    existing_col(raw_df, "cvss_vector"),
)

# vulnerabilities[] 또는 affected[] 배열, identifiers[] 배열은 원형 그대로 다음 단계(affected/alias build)로 넘긴다.
vulnerabilities_raw = coalesce_existing(raw_df, ["vulnerabilities", "affected"])
identifiers_raw = existing_array_col(raw_df, "identifiers")

# valid_from 우선순위: updated_at -> published_at -> ingested_at (02_Bronze_Silver_저장구조.md 표 1)
valid_from = F.coalesce(updated_at, published_at, F.lit(INGESTED_AT))

dq_flags = flags_array(
    F.when(cvss_score.isNull(), F.lit("MISSING_CVSS")),
    F.when(description.isNull(), F.lit("MISSING_DESCRIPTION")),
    F.when(F.size(F.coalesce(cwe_ids, F.array())) == 0, F.lit("MISSING_CWE")),
)


In [ ]:
normalized_df = raw_df.select(
    ghsa_id.alias("ghsa_id"),
    summary.alias("summary"),
    description.alias("description"),
    severity.alias("severity"),
    published_at.alias("published_at"),
    updated_at.alias("updated_at"),
    withdrawn_at.alias("withdrawn_at"),
    html_url.alias("html_url"),
    cwe_ids.alias("cwe_ids"),
    cvss_score.alias("cvss_score"),
    cvss_vector.alias("cvss_vector"),
    dq_flags.alias("dq_flags"),
    valid_from.alias("valid_from"),
    vulnerabilities_raw.alias("vulnerabilities_raw"),
    identifiers_raw.alias("identifiers_raw"),
    F.lit("ghsa").alias("source"),
    F.col("batch_id"),
    F.col("ingested_at"),
).filter(F.col("ghsa_id").isNotNull())

write_overwrite(normalized_df, TBL["stg_ghsa_normalized"])

record_count = normalized_df.count()
log_batch_run("normalize_ghsa", "SUCCESS", record_count=record_count)
print(f"[normalize_ghsa] normalized={record_count} batch_id={batch_id}")
